In [ ]:
# ============================================================
# CELL 0 — INSTALL
# ============================================================

!pip install -q -U \
    "torchao>=0.15.0" \
    transformers \
    accelerate \
    sentencepiece

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.0 MB/s eta 0:00:00
Installation complete.


In [ ]:
# ============================================================
# CELL 1 — IMPORTS + ENVIRONMENT
# ============================================================

import os
import time
import shutil

from pathlib import Path

import torch
import transformers
import torchao

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TorchAoConfig,
)

from torchao.quantization import (
    Int8WeightOnlyConfig,
)


print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TorchAO:", torchao.__version__)

print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cpu
Transformers: 5.17.0
TorchAO: 0.18.0
CUDA available: False


In [ ]:
# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

SOURCE_MODEL = (
    "JayShah07/falconai-text-bullet-t5"
)


OUTPUT_DIR = Path(
    "/content/text-to-bullets-int8"
)


BULLET_TOKEN = "<BULLET>"

MAX_INPUT_LENGTH = 2048

MAX_NEW_TOKENS = 256


print("Source:")
print(SOURCE_MODEL)

print("\nOutput:")
print(OUTPUT_DIR)

Source:
JayShah07/falconai-text-bullet-t5

Output:
/content/text-to-bullets-int8


In [ ]:
# ============================================================
# CELL 3 — CLEAN OUTPUT DIRECTORY
# ============================================================

if OUTPUT_DIR.exists():

    shutil.rmtree(
        OUTPUT_DIR
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Clean output directory:",
    OUTPUT_DIR
)

Clean output directory: /content/text-to-bullets-int8


In [ ]:
# ============================================================
# CELL 4 — LOAD TOKENIZER
# ============================================================

start = time.perf_counter()


tokenizer = AutoTokenizer.from_pretrained(
    SOURCE_MODEL,
)


tokenizer_load_seconds = (
    time.perf_counter()
    -
    start
)


print(
    "Tokenizer:",
    tokenizer.__class__.__name__
)

print(
    "Load time:",
    round(
        tokenizer_load_seconds,
        3
    ),
    "seconds"
)

print(
    "Vocabulary:",
    len(tokenizer)
)

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

Tokenizer: T5Tokenizer
Load time: 3.931 seconds
Vocabulary: 32101


In [ ]:
# ============================================================
# CELL 5 — VERIFY SPECIAL BULLET TOKEN
# ============================================================

bullet_token_id = (
    tokenizer.convert_tokens_to_ids(
        BULLET_TOKEN
    )
)


print(
    "BULLET token:",
    BULLET_TOKEN
)

print(
    "BULLET ID:",
    bullet_token_id
)

print(
    "UNK ID:",
    tokenizer.unk_token_id
)


assert (
    bullet_token_id
    !=
    tokenizer.unk_token_id
), (
    "<BULLET> is missing from tokenizer."
)


print(
    "\n<BULLET> token verified."
)

BULLET token: <BULLET>
BULLET ID: 32100
UNK ID: 2

<BULLET> token verified.


In [ ]:
# ============================================================
# CELL 6 — TORCHAO INT8 WEIGHT-ONLY CONFIG
# ============================================================

int8_config = (
    Int8WeightOnlyConfig()
)


quantization_config = (
    TorchAoConfig(
        quant_type=int8_config
    )
)


print(
    "Quantization:"
)

print(
    quantization_config
)

Quantization:
TorchAoConfig(quant_method=<QuantizationMethod.TORCHAO: 'torchao'>, quant_type=Int8WeightOnlyConfig(group_size=None, granularity=PerRow(dim=-1), set_inductor_config=True, version=2), modules_to_not_convert=None, include_input_output_embeddings=False, untie_embedding_weights=False)


In [ ]:
# ============================================================
# CELL 7 — LOAD SOURCE MODEL AS TORCHAO INT8
# ============================================================

print(
    "Loading and quantizing model..."
)


start = time.perf_counter()


model = (
    AutoModelForSeq2SeqLM
    .from_pretrained(

        SOURCE_MODEL,

        quantization_config=(
            quantization_config
        ),

        device_map="cpu",

        dtype=torch.bfloat16,
    )
)


model.eval()


load_seconds = (
    time.perf_counter()
    -
    start
)


print(
    "\nLoaded + quantized in:",
    round(
        load_seconds,
        3
    ),
    "seconds"
)


print(
    "Architecture:",
    model.__class__.__name__
)


print(
    "Encoder-decoder:",
    model.config.is_encoder_decoder
)


assert (
    model.config.is_encoder_decoder
)

Loading and quantizing model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]


Loaded + quantized in: 10.003 seconds
Architecture: T5ForConditionalGeneration
Encoder-decoder: True


In [ ]:
# ============================================================
# CELL 8 — ROBUSTLY VERIFY TORCHAO INT8 WEIGHT-ONLY
# ============================================================

import torch


print("=" * 80)
print("TORCHAO INT8 VERIFICATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. MODEL FOOTPRINT
# ------------------------------------------------------------

footprint_mb = (
    model.get_memory_footprint()
    / 1024**2
)

print(
    f"\nModel footprint: {footprint_mb:.2f} MB"
)


# ------------------------------------------------------------
# 2. INSPECT WEIGHT REPRESENTATIONS
# ------------------------------------------------------------

quantized_weights = []

normal_weights = []


for name, module in model.named_modules():

    weight = getattr(
        module,
        "weight",
        None,
    )

    if weight is None:
        continue

    try:

        weight_type = type(weight).__name__

        dtype = str(
            getattr(
                weight,
                "dtype",
                "unknown",
            )
        )

        representation = repr(
            type(weight)
        )


        # TorchAO representations have changed between versions.
        # Check several known indicators rather than relying on
        # one exact class name.

        is_quantized = any(
            marker in representation.lower()
            for marker in [
                "affinequantized",
                "linearactivationquantized",
                "quantized",
                "int8",
                "torchao",
            ]
        )


        if is_quantized:

            quantized_weights.append(
                {
                    "name": name,
                    "type": weight_type,
                    "dtype": dtype,
                    "class": representation,
                }
            )

        else:

            normal_weights.append(
                {
                    "name": name,
                    "type": weight_type,
                    "dtype": dtype,
                }
            )

    except Exception as exc:

        print(
            "Inspection skipped:",
            name,
            repr(exc),
        )


# ------------------------------------------------------------
# 3. PRINT FIRST QUANTIZED WEIGHTS
# ------------------------------------------------------------

print(
    "\nDetected quantized weights:",
    len(quantized_weights),
)


for item in quantized_weights[:15]:

    print(
        "\n",
        item["name"],
    )

    print(
        "  type :",
        item["type"],
    )

    print(
        "  dtype:",
        item["dtype"],
    )

    print(
        "  class:",
        item["class"],
    )


# ------------------------------------------------------------
# 4. SHOW SOME NON-QUANTIZED WEIGHTS TOO
# ------------------------------------------------------------

print(
    "\nNon-quantized weights:",
    len(normal_weights),
)


for item in normal_weights[:5]:

    print(
        item["name"],
        "->",
        item["type"],
        item["dtype"],
    )


# ------------------------------------------------------------
# 5. INSPECT QUANTIZATION CONFIG
# ------------------------------------------------------------

print(
    "\nModel quantization config:"
)

print(
    getattr(
        model.config,
        "quantization_config",
        None,
    )
)


# ------------------------------------------------------------
# 6. FINAL SANITY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)

if quantized_weights:

    print("✓ TorchAO quantized tensors detected.")

else:

    print(
        "⚠ No quantized tensor subclass was detected by type inspection."
    )


print(
    f"Measured model footprint: {footprint_mb:.2f} MB"
)


# Your previously validated TorchAO W8 model was ~146.7 MB.
# Use a broad sanity range because versions/configuration can
# introduce some representation differences.

if footprint_mb < 180:

    print(
        "✓ Footprint is consistent with a compressed/quantized model."
    )

else:

    print(
        "⚠ Footprint is too close to an unquantized FP32/BF16 model."
    )


print("=" * 80)


# ------------------------------------------------------------
# HARD ASSERTION
# ------------------------------------------------------------
#
# Do NOT require a specific TorchAO Python class name.
# Instead require:
#
#   quantized representation detected
#             OR
#   strongly reduced model footprint
#
# ------------------------------------------------------------

assert (
    len(quantized_weights) > 0
    or
    footprint_mb < 180
), (
    "Model does not appear to be TorchAO INT8 weight-only. "
    "Check Cell 6/7 quantization configuration."
)

TORCHAO INT8 VERIFICATION

Model footprint: 115.38 MB

Detected quantized weights: 96

 encoder.block.0.layer.0.SelfAttention.q
  type : Int8Tensor
  dtype: torch.bfloat16
  class: <class 'torchao.quantization.Int8Tensor'>

 encoder.block.0.layer.0.SelfAttention.k
  type : Int8Tensor
  dtype: torch.bfloat16
  class: <class 'torchao.quantization.Int8Tensor'>

 encoder.block.0.layer.0.SelfAttention.v
  type : Int8Tensor
  dtype: torch.bfloat16
  class: <class 'torchao.quantization.Int8Tensor'>

 encoder.block.0.layer.0.SelfAttention.o
  type : Int8Tensor
  dtype: torch.bfloat16
  class: <class 'torchao.quantization.Int8Tensor'>

 encoder.block.0.layer.1.DenseReluDense.wi
  type : Int8Tensor
  dtype: torch.bfloat16
  class: <class 'torchao.quantization.Int8Tensor'>

 encoder.block.0.layer.1.DenseReluDense.wo
  type : Int8Tensor
  dtype: torch.bfloat16
  class: <class 'torchao.quantization.Int8Tensor'>

 encoder.block.1.layer.0.SelfAttention.q
  type : Int8Tensor
  dtype: torch.bfloat16
  

In [ ]:
# ============================================================
# CELL 9 — MODEL MEMORY FOOTPRINT
# ============================================================

model_size_mb = (
    model.get_memory_footprint()
    /
    1024**2
)


print(
    "TorchAO INT8 model footprint:"
)

print(
    round(
        model_size_mb,
        2
    ),
    "MB"
)

TorchAO INT8 model footprint:
115.38 MB


In [ ]:
# ============================================================
# CELL 10 — TASK PROMPT
# ============================================================

TASK_INSTRUCTION = """
Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".

Text:
""".strip()

In [ ]:
# ============================================================
# CELL 11 — SANITY TEST
# ============================================================

TEST_TEXT = """
Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
Operating profit increased 8% to $620 million, although operating margin
declined from 17.2% to 14.8%. The company added 1.3 million customers
during the quarter and raised full-year revenue guidance from $16 billion
to $17.5 billion. Management warned that European demand weakened in July.
""".strip()


encoder_text = (
    TASK_INSTRUCTION
    +
    "\n"
    +
    TEST_TEXT
)


inputs = tokenizer(

    encoder_text,

    return_tensors="pt",

    max_length=MAX_INPUT_LENGTH,

    truncation=True,
)


print(
    "Input tokens:",
    inputs["input_ids"].shape[-1]
)

Input tokens: 279


In [ ]:
# ============================================================
# CELL 12 — VERIFY INT8 MODEL OUTPUT
# ============================================================

start = time.perf_counter()


with torch.inference_mode():

    output_ids = model.generate(

        **inputs,

        max_new_tokens=MAX_NEW_TOKENS,

        do_sample=False,

        num_beams=1,

        no_repeat_ngram_size=3,

        use_cache=True,
    )


latency = (
    time.perf_counter()
    -
    start
)


raw_output = tokenizer.decode(

    output_ids[0],

    skip_special_tokens=False,
)


print(
    "RAW OUTPUT:\n"
)

print(
    raw_output
)


print(
    "\nLatency:",
    round(
        latency,
        3
    ),
    "seconds"
)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW OUTPUT:

<pad><BULLET> Acme reported quarterly revenue of $4.2 billion, up 12% year over year.<BULLET> Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.<BULLET> The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.<BULLET> Management warned that European demand weakened in July.</s>

Latency: 5.478 seconds


In [ ]:
# ============================================================
# CELL 13 — SAVE TOKENIZER
# ============================================================

tokenizer.save_pretrained(
    OUTPUT_DIR
)


print(
    "Tokenizer saved."
)

Tokenizer saved.


In [ ]:
# ============================================================
# CELL 14 — SAVE TORCHAO INT8 MODEL
# ============================================================

print(
    "Saving INT8 model..."
)


start = time.perf_counter()


model.save_pretrained(
    OUTPUT_DIR
)


save_seconds = (
    time.perf_counter()
    -
    start
)


print(
    "Saved in:",
    round(
        save_seconds,
        3
    ),
    "seconds"
)

Saving INT8 model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved in: 0.355 seconds


In [ ]:
model.save_pretrained(
    OUTPUT_DIR,
    safe_serialization=False,
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# SHOW FILE SIZES INTELLIGENTLY
# ============================================================

for path in sorted(OUTPUT_DIR.rglob("*")):

    if not path.is_file():
        continue

    size_bytes = path.stat().st_size

    if size_bytes >= 1024**2:
        size = f"{size_bytes / 1024**2:.2f} MB"

    elif size_bytes >= 1024:
        size = f"{size_bytes / 1024:.2f} KB"

    else:
        size = f"{size_bytes} bytes"

    print(
        f"{path.name:<35} {size:>12}"
    )

config.json                              2.06 KB
generation_config.json                 877 bytes
model.safetensors                       73.78 MB
tokenizer.json                           2.31 MB
tokenizer_config.json                    2.45 KB


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/text-to-bullets-int8",
    "zip",
    "/content",
    "text-to-bullets-int8",
)

files.download(
    "/content/text-to-bullets-int8.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>